<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 1a — Introduction à la biologie moleculaire et préparation des données
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Comprendre la différence entre ADN codant et non-codant, et formuler la tâche : prédire si une fenêtre de taille 200 est codante<br>
    - Explorer les fichiers de donné bruts et voir comment on en extrait des fenêtres étiquetées<br>
    - Construire les splits train/val/test avec build_dataset.py et vérifier l'équilibre des classes<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** BOSSA Chabel
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Introduction**


1. ADN codant vs. non-codant: 
Un génome bactérien est une seule et longue chaîne d'ADN. Une partie seulement de cette
chaîne **code réellement pour une protéine** (une *CDS*, séquence codante — lue par
triplets appelés codons, chacun spécifiant un acide aminé). Le reste est **non-codant** :
régions intergéniques, séquences régulatrices, etc.

2. Notre tâche cette semaine : **étant donné une courte fenêtre d'ADN, prédire si elle se
trouve à l'intérieur d'une CDS ou non.**

3. C'est un problème réel et utile — la recherche de gènes (*gene finding*) est une tâche
fondamentale en bioinformatique, et elle est binaire, ce qui la rend abordable pour un
premier projet.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day1/assets/illustration0.png"/>

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Regardons d'abord les fichiers bruts**

Pas de prétraitement pour l'instant. Chaque génome dans `data/raw/` est une
paire de fichiers :
- un fichier **FASTA** : la séquence brute du génome (contient l'ADN)
- un fichier **GFF** : le fichier d'annotation. Il nous donne le sens de chaque sequence dans la donnée inital. Regardons les deux directement.

In [6]:
 
import sys
from pathlib import Path

# Si le notebook tourne depuis la racine du projet
if Path("2-data/raw").exists():
    RAW_DIR = "2-data/raw"
    if "day1/src" not in sys.path:
        sys.path.append("day1/src")
# Si le notebook tourne depuis le dossier day1/
else:
    RAW_DIR = "../2-data/raw"
    if "src" not in sys.path:
        sys.path.append("src")

print(f"Dossier raw utilisé : {RAW_DIR}")


Dossier raw utilisé : 2-data/raw


In [7]:
# TODO : ouvrez le fichier FASTA ci-dessous et affichez ses 6 premières lignes
# indice : with open(...) as f: for _ in range(6): print(f.readline().rstrip())
fasta_path = f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.fasta"
with open(fasta_path) as f:
    for _ in range(6):
        print(f.readline().rstrip())  # TODO : remplacez ... par la ligne à afficher

>CP003195.1 Propionibacterium acnes TypeIA2 P.acn33, complete genome
CTAGCGTTCAGGGGAGTGGTACATCGTGGGTAGCTTGTCACACCGACTGTGGAAAACTGTGTGGACAACTCTCGACAACT
TCTAGAGAACAGGTGGTGGAATGTCCGACACACCGTTCGGCGACGCCGACCACCCTCGTCCCGCGCCGATCCATCCGGAT
GCTGTATTGCCCCCGCCGATGAGTTCACAGTCGGCTGATAATGATCCCACTGAGGCCCTCAATGAAGCTTGGACCAACAT
CCTTACGAAGGTTTCGAAACCTAATCGCGCGTGGTTATCCAACACTACCCCGGTGACGATGCACTCTTCTACGGCGATGG
TTGCCGTTCCCAACGAATTTGCCCGTGACCGTCTTGAATCAAAGATGCGTTACGAACTAGAAGAACTCCTCTCTGACCAT


In [ ]:
# TODO : lisez le fichier GFF, ignorez les lignes de commentaire (qui commencent par '#'),
# puis affichez les 8 premières lignes restantes. Repérez les lignes de type CDS
# (colonnes : seqid, source, type, start, end, score, strand, phase, attributes).
gff_path = f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.gff"
with open(gff_path) as f:
    lines = [line for line in f if not line.startswith('#')]  

#### **Des fichiers bruts à un jeu de données étiqueté**

`src/build_dataset.py` est le script fourni qui transforme cela en quelque chose sur lequel
on peut entraîner un modèle. Pour chaque génome, il :

1. Lit tous les intervalles CDS du GFF (`type == "CDS"`, colonnes `start`/`end`/`strand`).
2. Fait glisser des fenêtres de longueur fixe (de taille 200) **à l'intérieur** des régions CDS →
   étiquette `1` (codant). Si la CDS est sur le brin `-`, la fenêtre est d'abord retournée
   en complément inverse, pour que le modèle voie toujours la séquence dans l'orientation
   codante.
3. Calcule les **intervalles** entre les régions CDS (tout ce qui n'est couvert par aucune
   CDS) et fait glisser les mêmes fenêtres dessus → étiquette `0` (non-codant).
4. Équilibre les classes par génome, mélange, et écrit un CSV par split.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day1/assets/illustration1.png"/>

Il a déjà été exécuté une fois (`data/processed/{train,val,test}.csv`
existent). Si vous voulez changer la taille de fenêtre ou le ré-échantillonnage,
relancez-le :

> ```bash
> python ../../code/src/build_dataset.py --raw_dir ../../data/raw --out_dir ../../data/processed --window 200 --stride 200
```

In [6]:
# N_WINDOWS fenêtres par split, équilibrées codant/non-codant : le MÊME budget
# que les embeddings Evo2 du Jour 2 (comparaison juste au Jour 4)
N_WINDOWS = 4000
from data import load_all

# Définition automatique du dossier des CSV traités ("2-data/processed")
PROCESSED_DIR = RAW_DIR.replace("raw", "processed")

# Charge les 3 splits (train, val, test)
splits = load_all(PROCESSED_DIR, max_rows=N_WINDOWS)

# Affiche pour chaque split sa taille (shape) et le nombre de 0 et 1 (class balance)
for name, df in splits.items():
    print(name, df.shape, "class balance:", df['label'].value_counts().to_dict())

# Affiche un aperçu du jeu d'entraînement
splits["train"].head()


train (4000, 9) class balance: {0: 2000, 1: 2000}
val (4000, 9) class balance: {0: 2000, 1: 2000}
test (4000, 9) class balance: {0: 2000, 1: 2000}


,id,split,organism,seqid,start,end,strand,label,sequence
0,train_16201,train,GCA_020683225.1_ASM2068322v1,CP085642.1,1276108,1276308,+,0,TATTTTGTAGGCCTGATAAGCGCAGCGCATCAGGCAACACGTCTGC...
1,train_5042,train,GCA_002863665.1_ASM286366v1,CP024056.1,313998,314198,+,1,GCTGGTGACCAAGCCGAAGAACTACGATCAGGTTCCGGCGAACAAA...
2,train_37631,train,GCA_002442855.1_ASM244285v1,CP017306.1,1210287,1210487,+,0,CGCACCACCTCCGGCATTGTTGTCGAAATACAATGGCTCGCCGCCC...
3,train_13606,train,GCA_039645065.1_ASM3964506v1,CP141690.1,1484519,1484719,+,1,TGGAATTTCACCCCGTAAAATTTGAAGAGCTTGCTCCTGATGCAAT...
4,train_24830,train,GCA_002277935.1_ASM227793v1,CP022930.1,2684271,2684471,-,1,ACGTAATCGATTTTGGTGCTTTCGTCGATATCGGGGTCAAACAAGA...


In [ ]:
# N_WINDOWS fenêtres par split, équilibrées codant/non-codant : le MÊME budget
# que les embeddings Evo2 du Jour 2 (comparaison juste au Jour 4)
N_WINDOWS = 4000
 
from data import load_all

splits = ...
for name, df in splits.items():
    print(name, ..., "class balance:", ...)

splits["train"].head()

#### **Point de contrôle**

Vous devriez avoir des DataFrames `train`, `val`, `test`, chacun avec une colonne
`sequence` (la fenêtre brute de 200 pb) et une colonne `label` (1 = codant, 0 = non-codant),
à peu près équilibrées entre les classes.

Suite : `01_kmer_and_cnn_baselines.ipynb` — construire les premiers modèles à partir de ces
données.

*Bloqué ? La version complète est dans `solution/00_biology_intro_and_data_setup.ipynb`.*

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 1a</div>
      <div>Prochaine &eacute;tape &rarr; <code>01_kmer_and_cnn_baselines.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>